<a href="https://colab.research.google.com/github/k24015887/group_project_1/blob/main/Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!mkdir -p interpolation_methods metrics plots main utils #This creates a folder inside on the content folder for each .py file

In [18]:
!touch utils/__init__.py
!touch interpolation_methods/__init__.py
!touch metrics/__init__.py
!touch plots/__init__.py
#This treats all folders as packages

In [31]:
%%writefile utils/utils.py
import numpy as np
from sklearn.model_selection import KFold

def create_grid_from_coords(coords, num_points):
  """
  Creates a grid covering the bounding box of the provided coordinates
  The coordinates will be the same for every one but the number of points can vary depending on the method

  Parameters:
  - coords: array of shape (n_samples, 3) with [x, y, z] coordinates
  - num_points: number of grid points per axis (resolution)

  Returns:
  - grid_x, grid_y, grid_z: arrays defining the grid (used for plotting)
  """
  min_x, max_x = np.min(coords[:, 0]), np.max(coords[:, 0])
  min_y, max_y = np.min(coords[:, 1]), np.max(coords[:, 1])
  min_z, max_z = np.min(coords[:, 2]), np.max(coords[:, 2])

  grid_x = np.linspace(min_x, max_x, num_points)
  grid_y = np.linspace(min_y, max_y, num_points)
  grid_z = np.linspace(min_z, max_z, num_points)
  return grid_x, grid_y, grid_z

def run_cross_validation(interpolator_class, coords, band_values, grid, num_folds=10):
  from metrics.metrics import extracted_interpolated_values, calculate_metrics
  kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
  cv_metrics = []
  cv_timings = []
  for train_index, test_index in kf.split(coords):
    train_coords = coords[train_index]
    train_values = band_values[train_index]
    test_coords = coords[test_index]
    test_values = band_values[test_index]

    #Instance of interpolator using training data:
    interpolator = interpolator_class(train_coords[:, 0], train_coords[:,1], train_coords[:,2],
                                          train_values, grid[0], grid[1], grid[2])
    predicted_values, duration = method(train_coords, train_values, test_coords, timing=True)
    cv_timings.append(duration)
    scores = calculate_metrics(predicted_values, test_values)
    cv_metrics.append(scores)
  return cv_metrics, cv_timings


Overwriting utils/utils.py


In [30]:
%%writefile plots/plots.py

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # required for 3D plots
from utils.utils import create_grid_from_coords
#======================================3D Visualisation===============================
def plot_3d(original_coords, original_values, predicted_values, vmin, vmax, method, grid=None, num_points=None):
  """
  Creates a 3D visualisation comparing original sparse data and interpolated volume

  Parameters:
  - original_coords: array of shape (n_samples, 3) with [x, y, z] coordinates
  - original_values: array of shape (n_samples, ) of values (e.g., power estimates) for each coordinate
  - predicted_values: 3D array of shape (n_points, n_points, n_points) with interpolated values
  - vmin,vmax: colour limits for plotting
  - method_name: string indicating the interpolation method to plot
  - grid: Optional tuple or calling function in utils
  """
  if grid is None:
    grid_x, grid_y, grid_z = create_grid_from_coords(original_coords, num_points)
  else:
    grid_x, grid_y, grid_z = grid

  #Create a mesh gid for the interpolation grid
  X, Y, Z = np.meshgrid(grid_x, grid_y, grid_z, indexing='ij')

  # Downsample the grid and interpolated volume for visualization
  step = 4  # adjust this as needed
  X_sample = X[::step, ::step, ::step].flatten()
  Y_sample = Y[::step, ::step, ::step].flatten()
  Z_sample = Z[::step, ::step, ::step].flatten()
  values_sample = predicted_values[::step, ::step, ::step].flatten()

  # Create a figure with two subplots for side-by-side comparison
  fig = plt.figure(figsize=(12, 6))

  # Subplot 1: Original sparse data in 3D
  ax1 = fig.add_subplot(121, projection='3d')
  p1 = ax1.scatter(original_coords[:, 0], original_coords[:, 1], original_coords[:, 2],
                    c=original_values, cmap='turbo', vmin=vmin, vmax=vmax)
  ax1.set_title("Original SEEG Data")
  ax1.set_xlabel("X")
  ax1.set_ylabel("Y")
  ax1.set_zlabel("Z")
  fig.colorbar(p1, ax=ax1, shrink=0.5)

  # Subplot 2: 3D downsampled interpolated data
  ax2 = fig.add_subplot(122, projection='3d')
  p2 = ax2.scatter(X_sample, Y_sample, Z_sample,
                    c=values_sample, cmap='turbo', vmin=vmin, vmax=vmax)
  ax2.set_title(f"3D Interpolated Data ({method})")
  ax2.set_xlabel("X")
  ax2.set_ylabel("Y")
  ax2.set_zlabel("Z")
  fig.colorbar(p2, ax=ax2, shrink=0.5)

  plt.tight_layout()
  plt.show()

#======================================2D Visualisation========================================
def plot_2d_slice(original_coords, original_values, predicted_values, vmin, vmax, method, grid=None):
    """
    Plots a 2D comparison of the original data and interpolated results for the middle z-slice.

    Parameters:
      - original_coords: (n, 3) array of original coordinates.
      - original_values: (n,) array of values for each coordinate.
      - interpolated_volume: 3D array of interpolated values (shape: [nz, ny, nx]).
      - vmin, vmax: Color limits.
      - method_name: Name of the interpolation method.
      - grid: Optional tuple (grid_x, grid_y, grid_z). If None, the grid is computed from original_coords.
    """
    if grid is None:
        grid_x, grid_y, grid_z = create_grid_from_coords(original_coords)
    else:
        grid_x, grid_y, grid_z = grid

    # Determine the middle slice index and corresponding z value
    mid_idx = predicted_values.shape[0] // 2
    mid_z = grid_z[len(grid_z) // 2]

    # Filter the original data to only include points near the middle z value.
    tolerance = (grid_z[1] - grid_z[0]) / 2
    indices = np.where(np.abs(original_coords[:, 2] - mid_z) < tolerance)[0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Plot original data for the middle slice
    sc = ax1.scatter(original_coords[indices, 0], original_coords[indices, 1],
                     c=original_values[indices], cmap='turbo', vmin=vmin, vmax=vmax)
    ax1.set_title(f"Original Data (z ≈ {mid_z:.2f})")
    ax1.set_xlabel("X")
    ax1.set_ylabel("Y")
    plt.colorbar(sc, ax=ax1, label="Power Estimate")

    # Plot the 2D interpolated slice from the volume
    im = ax2.imshow(predicted_values[mid_idx, :, :],
                    extent=(np.min(grid_x), np.max(grid_x),
                            np.min(grid_y), np.max(grid_y)),
                    origin='lower', cmap='turbo', vmin=vmin, vmax=vmax)
    ax2.set_title(f"2D Interpolated Slice ({method})")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    plt.colorbar(im, ax=ax2, label="Interpolated Value")

    plt.tight_layout()
    plt.show()


Overwriting plots/plots.py


In [37]:
%%writefile interpolation_methods/interpolation_methods.py
from pykrige.ok3d import OrdinaryKriging3D
from sklearn.neighbors import KNeighborsRegressor
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

#In the main.py files we can then access each method individually by having them each in a different function better than a class

#=========================================Kriging Functions=========================================================
#-----------------------------------------Gaussian Kriging----------------------------------------------------------
def Kriging_Gaussian(coords, values, new_coords, timing=False):
  """
  Interpolates values using Gaussian Kriging
  Parameters:
  - coords (array): original coordinates x, y, and z (shape: [n_samples, 3])(training set)
  - values (array): corresponding power values at these coordinates (shape: [n_samples])
  - new_coords (array): new coordinates for interpolation (test set)
  - timing (bool): if True, it will return the time it takes to perform interpolation
  Returns:
  - interpolated_values (array): interpolated values at new_coords (shape: [n_new_samples])
  - duration (float): if timing is True, it will return the time it takes to perform interpolation
  """
  start_time = time.time()
  x_new, y_new, z_new = new_coords[:, 0], new_coords[:, 1], new_coords[:, 2]

  OK = OrdinaryKriging3D(
    coords[:, 0], coords[:, 1], coords[:, 2],
    values,
    variogram_model='gaussian',
    verbose=False,
    enable_plotting=False
  )
  predicted_values, ss = OK.execute('points', x_new.tolist(), y_new.tolist(), z_new.tolist())
  duration = time.time() - start_time
  predicted_values = np.array(predicted_values)
  return (predicted_values, duration) if timing else predicted_values
#-----------------------------------Spherical Kriging----------------------------------------------------
def Kriging_Spherical(coords, values, new_coords, timing=False):
  """
  Interpolates values using Spherical Kriging
  Parameters:
  - coords (array): original coordinates x, y, and z (shape: [n_samples, 3])(training set)
  - values (array): corresponding power values at these coordinates (shape: [n_samples])
  - new_coords (array): new coordinates for interpolation (test set)
  - timing (bool): if True, it will return the time it takes to perform interpolation
  Returns:
  - interpolated_values (array): interpolated values at new_coords (shape: [n_new_samples])
  - duration (float): if timing is True, it will return the time it takes to perform interpolation
  """
  start_time = time.time()
  x_new, y_new, z_new = new_coords[:, 0], new_coords[:, 1], new_coords[:, 2]

  OK = OrdinaryKriging3D(
    coords[:, 0], coords[:, 1], coords[:, 2],
    values,
    variogram_model='spherical',
    verbose=False,
    enable_plotting=False
  )
  predicted_values, ss = OK.execute('points', x_new.tolist(), y_new.tolist(), z_new.tolist())
  duration = time.time() - start_time
  predicted_values = np.array(predicted_values)
  return (predicted_values, duration) if timing else predicted_values
#-------------------------------Exponential Kriging----------------------------------------------------
def Kriging_Exponential(coords, values, new_coords, timing=False):

  """
  Interpolates values using Exponential Kriging
  Parameters:
  - coords (array): original coordinates x, y, and z (shape: [n_samples, 3])(training set)
  - values (array): corresponding power values at these coordinates (shape: [n_samples])
  - new_coords (array): new coordinates for interpolation (test set)
  - timing (bool): if True, it will return the time it takes to perform interpolation
  Returns:
  - interpolated_values (array): interpolated values at new_coords (shape: [n_new_samples])
  - duration (float): if timing is True, it will return the time it takes to perform interpolation
  """
  start_time = time.time()
  x_new, y_new, z_new = new_coords[:, 0], new_coords[:, 1], new_coords[:, 2]

  OK = OrdinaryKriging3D(
    coords[:, 0], coords[:, 1], coords[:, 2],
    values,
    variogram_model='exponential',
    verbose=False,
    enable_plotting=False
  )
  predicted_values, ss = OK.execute('points', x_new.tolist(), y_new.tolist(), z_new.tolist())
  duration = time.time() - start_time
  predicted_values = np.array(predicted_values)
  return (predicted_values, duration) if timing else predicted_values

  # -------------------------------- Linear Interpolation ---------------------------------------#
def Linear_Interpolation(coords, values, new_coords, timing=False):
    """
    Interpolates values using linear interpolation.
    Parameters:
    - coords (array): original coordinates x and y (shape: [n_samples, 2]) (training set)
    - values (array): corresponding power values at these coordinates (shape: [n_samples])
    - new_coords (array): new coordinates for interpolation (test set)
    - timing (bool): if True, returns the time taken for interpolation
    Returns:
    - interpolated_values (array): predicted values at new_coords (shape: [n_new_samples])
    - duration (float): if timing is True, returns the time taken for interpolation
    """
    start_time = time.time()

    predicted_values = griddata(coords, values, new_coords, method='linear')

    duration = time.time() - start_time
    return (predicted_values, duration) if timing else predicted_values

# -------------------------------- KNN Interpolation Function ------------------------------------#
def KNN_Interpolation(coords, values, new_coords, timing=False):
    """
    Interpolates values using KNN.
    Parameters:
    - coords (array): original coordinates x and y (shape: [n_samples, 2]) (training set)
    - values (array): corresponding power values at these coordinates (shape: [n_samples])
    - new_coords (array): new coordinates for interpolation (test set)
    - timing (bool): if True, returns the time taken for interpolation
    Returns:
    - interpolated_values (array): predicted values at new_coords (shape: [n_new_samples])
    - duration (float): if timing is True, returns the time taken for interpolation
    """
    start_time = time.time()

    knn = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn.fit(coords, values)
    predicted_values = knn.predict(new_coords)

    duration = time.time() - start_time
    return (predicted_values, duration) if timing else predicted_values



Overwriting interpolation_methods/interpolation_methods.py


In [44]:
%%writefile main/main.py
import os
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd
from pykrige.ok3d import OrdinaryKriging3D


#Import functions from the modules
from utils.utils import create_grid_from_coords, run_cross_validation
from interpolation_methods.interpolation_methods import Kriging_Gaussian, Kriging_Spherical, Kriging_Exponential, Linear_Interpolation, KNN_Interpolation
from metrics.metrics import calculate_metrics, extracted_interpolated_values
from plots.plots import plot_3d, plot_2d_slice

#Set file path for the SEEG data (We have to either create a shared google drive or a google colab bucket to include the data folder)
file_path = '/content/drive/MyDrive/data/seeg'

#Load coordinates (shape: [1772, 3])
coords_path = os.path.join(file_path, 'coords.npy')
coords = np.load(coords_path)

#Load band power data (a dictionary with keys)
band_powers_path = os.path.join(file_path, 'band_powers.npy')
band_powers = np.load(band_powers_path, allow_pickle=True).item()

# Choose a frequency band to interpolate (e.g., 'Delta')
#We can choose in the future if we want to manually choose frequency band or if we want it to loop through all values
band = 'Delta'
band_values = np.array(band_powers[band])
vmin = np.min(band_values)
vmax = np.max(band_values)

#Call the grid creator covering the full brain bounding box (need to talk all together about what we need num_points to be)
grid_x, grid_y, grid_z = create_grid_from_coords(coords, num_points=40) #I have put 40 for now because this is what works with my kriging without crashing

#Chosse interpolation method that we want to use
used_method = 'Gaussian Kriging' #just using one of mine as an example

if used_method == 'Gaussian Kriging':
  method = Kriging_Gaussian
  interpolator_class = Kriging_Gaussian
elif used_method == 'Spherical Kriging':
  method = Kriging_Spherical
  interpolator_class = Kriging_Spherical
elif used_method == 'Exponential Kriging':
  method = Kriging_Exponential
  interpolator_class = Kriging_Exponential
elif used_method == 'Linear Interpolation':
  method = Linear_Interpolation
  interpolator_class = Linear_Interpolation
elif used_method == 'KNN Interpolation':
  method = KNN_Interpolation
  interpolator_class = KNN_Interpolation
  #Add your methods please
cv_metrics, cv_timings = run_cross_validation(interpolator_class, coords, band_values, grid=(grid_x, grid_y, grid_z))
#Process the cross-validation metrics into a DataFrame
df_cv = pd.DataFrame(cv_metrics)
df_cv['Time (sec)'] = cv_timings

print("Cross-Validation Metrics per Fold:")
print(df_cv.to_string(index=False))

# Compute average and standard deviation across folds
avg_metrics = df_cv.mean()
std_metrics = df_cv.std()

print("\nAverage CV Metrics:")
print(avg_metrics.to_string())

print("\nStandard Deviation of CV Metrics:")
print(std_metrics.to_string())

#call the plotting functions:
plot_3d(coords, band_values, predicted_values, vmin, vmax, method, grid=(grid_x, grid_y, grid_z))
plot_2d_slice(coords, band_values, predicted_values, vmin, vmax, method, grid=(grid_x, grid_y, grid_z))

Overwriting main/main.py


In [22]:
#This creates the .py file
%%writefile metrics/metrics.py
# Whatever is under this will be loaded into the metrics.py
#To call this in the main.py file w ehave to import metrics.metrics.py

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def extracted_interpolated_values(OK_obj, x_coords, y_coords, z_coords):
    """
    Extracts interpolated values at provided coordinates using the given kriging object.

    Parameters:
      - OK_obj: An OrdinaryKriging3D object.
      - x_coords, y_coords, z_coords: Arrays of coordinates.

    Returns:
      - Array of predicted values.
    """
    interpolated_values = []
    for i in range(len(x_coords)):
        val, _ = OK_obj.execute('points', [x_coords[i]], [y_coords[i]], [z_coords[i]])
        interpolated_values.append(val[0])
    return np.array(interpolated_values)

def calculate_metrics(predicted_values, test_values, metric=None):
    """
    Compute evaluation metrics to evaluate interpoaltion performance
    Parameters:
      - pred_values: numpy array of predicted values
      - test_values: numpy array of ground truth values for comparison
      - metrics (list): list of metrics to compute
    Returns:
      - scores (dict): A dictionary with the computed metric scores.
    """
    mse = mean_squared_error(test_values, predicted_values)
    mae = mean_absolute_error(test_values, predicted_values)
    rmse = np.sqrt(mse)
    r2 = r2_score(test_values, predicted_values)
    scores = {"MSE": mse, "MAE": mae, "RMSE": rmse, "R²": r2}
    return scores


Overwriting metrics/metrics.py


In [45]:
!pip install pykrige
#If we go for google drive we have to have the following two lines of code:
from google.colab import drive
drive.mount('/content/drive')

!python -m main.main


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/main/main.py", line 55, in <module>
    cv_metrics, cv_timings = run_cross_validation(interpolator_class, coords, band_values, grid=(grid_x, grid_y, grid_z))
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/utils/utils.py", line 37, in run_cross_validation
    interpolator = interpolator_class(train_coords[:, 0], train_coords[:,1], train_coords[:,2],
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: Kriging_Gaussian() takes from 3 to 4 positional arguments but 7 were given
